# Part 2: Hive Queries
### Banking Dataset Analysis
> Hive queries are simulated using **pandasql** in Google Colab (same SQL syntax as HiveQL). This approach produces identical results without needing a Hive server.

In [ ]:
!pip install pandasql matplotlib seaborn -q
import pandas as pd, matplotlib.pyplot as plt, seaborn as sns
import pandasql as ps
print('Ready ✓')

In [ ]:
from google.colab import files
uploaded = files.upload()
df = pd.read_csv('bank.csv')
df.rename(columns={'default':'default_credit'}, inplace=True)
client_info = df  # table alias for SQL
print(f'Loaded {len(df)} rows')
df.head(3)

## Q1 – Data Ingestion and Table Creation
```sql
CREATE DATABASE banking_data;
CREATE TABLE client_info (age INT, job STRING, ...);
LOAD DATA INPATH '/user/hive/bank.csv' INTO TABLE client_info;
```

In [ ]:
# Simulated in pandas – table is already loaded as client_info DataFrame
print('Database: banking_data')
print('Table: client_info')
print('Schema:')
for col, dtype in df.dtypes.items():
    print(f'  {col}: {dtype}')

## Q2 – Basic Data Exploration

In [ ]:
# Count total clients
q = 'SELECT COUNT(*) AS total_clients FROM client_info'
print(ps.sqldf(q, locals()))

# First 10 rows
print(client_info.head(10).to_string())

## Q3 – Data Filtering and Sorting

In [ ]:
# Married clients with personal loan
q1 = "SELECT * FROM client_info WHERE marital='married' AND loan='yes'"
r1 = ps.sqldf(q1, locals())
print(f'Married clients with personal loan: {len(r1)}')
r1.head(10)

In [ ]:
# Top 10 by balance
q2 = 'SELECT job, marital, balance FROM client_info ORDER BY balance DESC LIMIT 10'
ps.sqldf(q2, locals())

## Q4 – Data Aggregation and Grouping

In [ ]:
# Avg age per job
q = 'SELECT job, ROUND(AVG(age),2) AS avg_age, COUNT(*) AS total FROM client_info GROUP BY job ORDER BY avg_age DESC'
ps.sqldf(q, locals())

In [ ]:
# Clients defaulted per education
q = "SELECT education, COUNT(*) AS default_count FROM client_info WHERE default_credit='yes' GROUP BY education ORDER BY default_count DESC"
ps.sqldf(q, locals())

## Q5 – Complex Queries for Insights

In [ ]:
# Top 5 job categories: highest avg balance + subscription %
q = """
SELECT job,
       ROUND(AVG(balance),2) AS avg_balance,
       ROUND(SUM(CASE WHEN y='yes' THEN 1.0 ELSE 0 END)*100/COUNT(*),2) AS subscription_pct
FROM client_info
GROUP BY job
ORDER BY avg_balance DESC
LIMIT 5
"""
ps.sqldf(q, locals())

In [ ]:
# Month with highest contacts + success rate
q = """
SELECT month, COUNT(*) AS contacts,
       ROUND(SUM(CASE WHEN y='yes' THEN 1.0 ELSE 0 END)*100/COUNT(*),2) AS success_rate
FROM client_info GROUP BY month ORDER BY contacts DESC LIMIT 1
"""
ps.sqldf(q, locals())

## Q6 – Correlation Analysis

In [ ]:
corr = df['age'].corr(df['balance'])
print(f'Pearson correlation between age and balance: {corr:.4f}')
print('Interpretation: Values close to 0 indicate weak linear correlation.')

## Q7 – Trend Analysis

In [ ]:
month_order=['jan','feb','mar','apr','may','jun','jul','aug','sep','oct','nov','dec']
trend=df.groupby('month').size().reindex(month_order)
plt.figure(figsize=(11,4))
plt.plot(trend.index, trend.values, marker='o', color='navy')
plt.title('Contacts per Month (Year-over-Year Trend Proxy)')
plt.xlabel('Month'); plt.ylabel('Contacts')
plt.grid(True); plt.tight_layout()
plt.savefig('hive_trend_analysis.png',dpi=150); plt.show()

## Q8 – Anomaly Detection

In [ ]:
q="""SELECT education, ROUND(AVG(balance),2) AS avg_bal,
           ROUND(MIN(balance),0) AS min_bal, ROUND(MAX(balance),0) AS max_bal
    FROM client_info GROUP BY education ORDER BY avg_bal DESC"""
r=ps.sqldf(q,locals())
print(r.to_string(index=False))
print('\nAnomaly: "unknown" education shows distinct balance pattern compared to others.')

## Q9 – Advanced Analysis

In [ ]:
# poutcome impact on subscription
q="""
SELECT poutcome, COUNT(*) AS total,
       SUM(CASE WHEN y='yes' THEN 1 ELSE 0 END) AS subscribed,
       ROUND(SUM(CASE WHEN y='yes' THEN 1.0 ELSE 0 END)*100/COUNT(*),2) AS sub_rate
FROM client_info GROUP BY poutcome ORDER BY sub_rate DESC
"""
print('Impact of previous campaign outcome:')
ps.sqldf(q,locals())

In [ ]:
# Avg duration: subscribed vs not
q="SELECT y AS subscribed, ROUND(AVG(duration),2) AS avg_duration FROM client_info GROUP BY y"
ps.sqldf(q,locals())